In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print("Environment siap.")

Environment siap.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
path = "/content/drive/MyDrive/collab/pantaupadi"

os.chdir(path)

print("File di folder ini:")
print(os.listdir())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = "/content/drive/MyDrive/collab/pantaupadi"

os.chdir(path)

print("File di folder ini:")
print(os.listdir())

File di folder ini:
['final_iklim.csv', 'padi_bulanan_bps', 'Dataset_Final_Emas_V1.csv', 'mergepadi.csv', 'luas_panen', 'full_dataset.csv', 'produktivias.csv', 'pantaupadi.ipynb']


In [ ]:
#merge data produksi padi

import pandas as pd
import glob
import re

print("=== TAHAP 1: MEMBACA DATA IKLIM (PATOKAN) ===")
df_iklim = pd.read_csv('final_iklim.csv')

df_iklim['ADM2_NAME'] = df_iklim['ADM2_NAME'].astype(str).str.upper().str.strip()
df_iklim['ADM2_NAME'] = df_iklim['ADM2_NAME'].str.replace(r'^(KABUPATEN\s+|KOTA\s+|KAB\.\s+)', '', regex=True)
df_iklim['ADM2_NAME'] = df_iklim['ADM2_NAME'].str.strip()
df_iklim['tahun'] = df_iklim['tahun'].astype(int)
df_iklim['bulan'] = df_iklim['bulan'].astype(int)
print(f"Data Iklim siap: {df_iklim.shape[0]} baris.")

print("\n=== TAHAP 2: EKSTRAKSI DATA BPS (DYNAMIC HEADER) ===")
month_map = {
    'januari': 1, 'februari': 2, 'maret': 3, 'april': 4,
    'mei': 5, 'juni': 6, 'juli': 7, 'agustus': 8,
    'september': 9, 'oktober': 10, 'november': 11, 'desember': 12
}

path_bps = "padi_bulanan_bps/*.xlsx"
file_list = glob.glob(path_bps)
all_bps_data = []

for file in file_list:
    # 1. Baca mentahan tanpa header
    df_raw = pd.read_excel(file, header=None)

    header_idx = None
    for i, row in df_raw.iterrows():
        row_str = row.astype(str).str.lower().str.strip().tolist()
        if 'januari' in row_str:
            header_idx = i
            break

    if header_idx is None:
        print(f"  -> Skip {file}: Kata 'Januari' tidak ditemukan sama sekali.")
        continue

    df_temp = df_raw.iloc[header_idx + 1:].copy()
    df_temp.columns = df_raw.iloc[header_idx]

    kolom_kab_asli = df_temp.columns[0]

    cari_tahun = re.findall(r'\d{4}', file)
    tahun_file = int(cari_tahun[-1]) if cari_tahun else 9999

    df_long = pd.melt(df_temp, id_vars=[kolom_kab_asli], var_name='nama_bulan', value_name='produksi_ton')
    df_long.rename(columns={kolom_kab_asli: 'ADM2_NAME'}, inplace=True)
    df_long['tahun'] = tahun_file

    df_long['nama_bulan'] = df_long['nama_bulan'].astype(str).str.lower().str.strip()
    df_long['bulan'] = df_long['nama_bulan'].map(month_map)

    df_long = df_long.dropna(subset=['bulan'])
    df_long = df_long.drop(columns=['nama_bulan'])
    all_bps_data.append(df_long)
    print(f"  -> Sukses ekstraksi file: {file} (Header otomatis ketemu di baris {header_idx})")

df_bps = pd.concat(all_bps_data, ignore_index=True)
df_bps['ADM2_NAME'] = df_bps['ADM2_NAME'].astype(str).str.upper().str.strip()
df_bps['ADM2_NAME'] = df_bps['ADM2_NAME'].str.replace(r'^(KABUPATEN\s+|KOTA\s+|KAB\.\s+)', '', regex=True)
df_bps['ADM2_NAME'] = df_bps['ADM2_NAME'].str.strip()
df_bps['tahun'] = df_bps['tahun'].astype(int)
df_bps['bulan'] = df_bps['bulan'].astype(int)
print(f"Data BPS berhasil digabung: {df_bps.shape[0]} baris.")

print("\n=== TAHAP 3: MERGING FINAL ===")
df_final = pd.merge(df_iklim, df_bps, on=['ADM2_NAME', 'tahun', 'bulan'], how='left')

nan_wajar = df_final[df_final['tahun'] > 2024]['produksi_ton'].isna().sum()
nan_gawat = df_final[df_final['tahun'] <= 2024]['produksi_ton'].isna().sum()

print(f"Jumlah NaN Wajar (2025 ke atas) : {nan_wajar} baris")
print(f"Jumlah NaN Gawat (2018 - 2024)  : {nan_gawat} baris")

if nan_gawat == 0:
    print("BINGO!!! Semua data BPS historis sukses menempel tanpa ada yang bolong!")
else:
    print("Masih ada data yang bocor di tahun historis.")

# Simpan
path_output = 'mergepadi.csv'
df_final.to_csv(path_output, index=False)
print(f"\nFile master siap di: {path_output}")

=== TAHAP 1: MEMBACA DATA IKLIM (PATOKAN) ===
Data Iklim siap: 3861 baris.

=== TAHAP 2: EKSTRAKSI DATA BPS (DYNAMIC HEADER) ===
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2022.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2019.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2024.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2020.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2023.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2021.xlsx (Header otomatis ketemu di baris 3)
  -> Sukses ekstraksi file: padi_bulanan_bps/Produksi Padi Menurut Kabupaten_Kota, 2018.xlsx (Hea

In [ ]:
import pandas as pd
import glob
import re

print("=== TAHAP 1: LOAD MASTER DATASET V1 ===")
path_master = '/mergepadi.csv'
df_master = pd.read_csv(path_master)
print(f"Master Dataset siap: {df_master.shape[0]} baris.")

print("\n=== TAHAP 2: EKSTRAKSI DATA LUAS PANEN (CSV VERSION) ===")
month_map = {
    'januari': 1, 'februari': 2, 'maret': 3, 'april': 4,
    'mei': 5, 'juni': 6, 'juli': 7, 'agustus': 8,
    'september': 9, 'oktober': 10, 'november': 11, 'desember': 12
}

path_luas = "luas_panen/*.csv"
file_list = glob.glob(path_luas)
all_luas_data = []

if not file_list:
    print("CRITICAL ERROR: Folder kosong! Pastikan file CSV lu udah ada di folder 'luas_panen_bps'.")
else:
    for file in file_list:

        df_raw = pd.read_csv(file, header=None)

        header_idx = None
        for i, row in df_raw.iterrows():
            row_str = row.astype(str).str.lower().str.strip().tolist()
            if 'januari' in row_str:
                header_idx = i
                break

        if header_idx is None:
            continue

        df_temp = df_raw.iloc[header_idx + 1:].copy()

        df_temp.columns = df_raw.iloc[header_idx].fillna('KAB_KOTA')

        kolom_kab_asli = df_temp.columns[0]
        cari_tahun = re.findall(r'\d{4}', file)
        tahun_file = int(cari_tahun[-1]) if cari_tahun else 9999

        df_long = pd.melt(df_temp, id_vars=[kolom_kab_asli], var_name='nama_bulan', value_name='luas_panen_hektar')
        df_long.rename(columns={kolom_kab_asli: 'ADM2_NAME'}, inplace=True)
        df_long['tahun'] = tahun_file

        df_long['nama_bulan'] = df_long['nama_bulan'].astype(str).str.lower().str.strip()
        df_long['bulan'] = df_long['nama_bulan'].map(month_map)

        df_long = df_long.dropna(subset=['bulan'])
        df_long = df_long.drop(columns=['nama_bulan'])
        all_luas_data.append(df_long)
        print(f"  -> Sukses ekstraksi file CSV: {file.split('/')[-1]}")

    df_luas = pd.concat(all_luas_data, ignore_index=True)
    df_luas['ADM2_NAME'] = df_luas['ADM2_NAME'].astype(str).str.upper().str.strip()
    df_luas['ADM2_NAME'] = df_luas['ADM2_NAME'].str.replace(r'^(KABUPATEN\s+|KOTA\s+|KAB\.\s+)', '', regex=True)
    df_luas['ADM2_NAME'] = df_luas['ADM2_NAME'].str.strip()

    df_luas['luas_panen_hektar'] = pd.to_numeric(
        df_luas['luas_panen_hektar'].astype(str).str.replace(',', ''),
        errors='coerce'
    )
    df_luas['tahun'] = df_luas['tahun'].astype(int)
    df_luas['bulan'] = df_luas['bulan'].astype(int)
    print(f"\nData Luas Panen siap di-merge: {df_luas.shape[0]} baris.")

    print("\n=== TAHAP 3: MERGE FINAL V2 ===")
    df_final_v2 = pd.merge(df_master, df_luas, on=['ADM2_NAME', 'tahun', 'bulan'], how='left')

    nan_gawat = df_final_v2[df_final_v2['tahun'] <= 2024]['luas_panen_hektar'].isna().sum()
    if nan_gawat == 0:
        print("PERFECT! Seluruh data Iklim, Produksi, dan Luas Panen resmi menyatu 100%.")
    else:
        print(f"WARNING: Ada {nan_gawat} baris luas panen yang gagal menempel di rentang 2018-2024.")

    path_output = 'full_dataset.csv'
    df_final_v2.to_csv(path_output, index=False)
    print(f"\nFile master V2 tersimpan di: {path_output}")

=== TAHAP 1: LOAD MASTER DATASET V1 ===
Master Dataset siap: 5037 baris.

=== TAHAP 2: EKSTRAKSI DATA LUAS PANEN (CSV VERSION) ===
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2022.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2020.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2018.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2019.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2021.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2024.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2023.csv
  -> Sukses ekstraksi file CSV: Luas Panen Padi Menurut Kabupaten_Kota, 2025.csv

Data Luas Panen siap di-merge: 3744 baris.

=== TAHAP 3: MERGE FINAL V2 ===

File master V2 tersimpan di: full_dataset.csv


Mulai Bekerja


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('full_dataset.csv')

df.head()

,ADM2_NAME,tahun,bulan,total_hujan,max_hth,nilai_normal_30th,anomali_persen,sifat_hujan_30th,kat_value,kat_normal,kat_anomaly_pct,produksi_ton,luas_panen_hektar
0,BANGKALAN,2018,1,451.514587,0,334.360256,135.038354,AN,392.434481,371.607650,105.605586,4739.48,785.0
1,BANGKALAN,2018,2,202.027545,2,250.459204,80.662855,BN,395.880555,378.690230,104.524388,46564.49,7716.0
2,BANGKALAN,2018,3,227.062383,1,243.218011,93.357553,N,395.380618,378.722207,104.334329,68927.85,11434.0
3,BANGKALAN,2018,4,97.913194,3,201.669351,48.551351,BN,376.930371,368.126476,102.167180,27021.81,4477.0
4,BANGKALAN,2018,5,30.727801,17,115.417852,26.623092,BN,313.243207,342.708690,91.000371,12847.5,2218.0


In [ ]:
#bersihin

df.columns = df.columns.str.strip()

df['wilayah'] = df['ADM2_NAME'].astype(str).str.strip().str.upper()

df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce')
df['bulan'] = pd.to_numeric(df['bulan'], errors='coerce')

df['tanggal'] = pd.to_datetime(
    df['tahun'].astype(int).astype(str) + '-' +
    df['bulan'].astype(int).astype(str).str.zfill(2) + '-01'
)

df['sifat_hujan'] = df['sifat_hujan_30th'].astype(str).str.strip().str.upper()

In [ ]:
numeric_cols = [
    'total_hujan',
    'max_hth',
    'nilai_normal_30th',
    'anomali_persen',
    'kat_value',
    'kat_normal',
    'kat_anomaly_pct',
    'produksi_ton',
    'luas_panen_hektar'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
#cleaning biar ga ada yang duplikat


def mode_or_first(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

master = (
    df.groupby(['wilayah', 'tahun', 'bulan'], as_index=False)
    .agg(
        hujan_mm=('total_hujan', 'mean'),
        HTH_hari=('max_hth', 'max'),
        hujan_normal_mm=('nilai_normal_30th', 'mean'),
        rasio_hujan_terhadap_normal=('anomali_persen', lambda x: np.nanmean(x) / 100),
        sifat_hujan=('sifat_hujan', mode_or_first),
        KAT_value=('kat_value', 'mean'),
        KAT_normal=('kat_normal', 'mean'),
        rasio_KAT_terhadap_normal=('kat_anomaly_pct', lambda x: np.nanmean(x) / 100),
        produksi_padi_bulanan=('produksi_ton', 'sum'),
        luas_panen_bulanan=('luas_panen_hektar', 'sum')
    )
)

master['tanggal'] = pd.to_datetime(
    master['tahun'].astype(int).astype(str) + '-' +
    master['bulan'].astype(int).astype(str).str.zfill(2) + '-01'
)

master.head()

,wilayah,tahun,bulan,hujan_mm,HTH_hari,hujan_normal_mm,rasio_hujan_terhadap_normal,sifat_hujan,KAT_value,KAT_normal,rasio_KAT_terhadap_normal,produksi_padi_bulanan,luas_panen_bulanan,tanggal
0,BANGKALAN,2018,1,451.514587,0,334.360256,1.350384,AN,392.434481,371.607650,1.056056,4739.48,785.0,2018-01-01
1,BANGKALAN,2018,2,202.027545,2,250.459204,0.806629,BN,395.880555,378.690230,1.045244,46564.49,7716.0,2018-02-01
2,BANGKALAN,2018,3,227.062383,1,243.218011,0.933576,N,395.380618,378.722207,1.043343,68927.85,11434.0,2018-03-01
3,BANGKALAN,2018,4,97.913194,3,201.669351,0.485514,BN,376.930371,368.126476,1.021672,27021.81,4477.0,2018-04-01
4,BANGKALAN,2018,5,30.727801,17,115.417852,0.266231,BN,313.243207,342.708690,0.910004,12847.50,2218.0,2018-05-01


In [ ]:
#sifat hujan score
def hujan_score_from_sifat_hujan(x):
    if x == 'BN':
        return 2
    elif x in ['N', 'AN']:
        return 0
    else:
        return np.nan

master['hujan_score'] = master['sifat_hujan'].apply(hujan_score_from_sifat_hujan)

In [ ]:
#hth score

q_hth_33 = master['HTH_hari'].quantile(0.33)
q_hth_66 = master['HTH_hari'].quantile(0.66)

def score_hth(x):
    if pd.isna(x):
        return np.nan
    elif x <= q_hth_33:
        return 0
    elif x <= q_hth_66:
        return 1
    else:
        return 2

master['HTH_score'] = master['HTH_hari'].apply(score_hth)

print(q_hth_33, q_hth_66)

3.0 11.0


In [ ]:
#kat score

q_kat_33 = master['rasio_KAT_terhadap_normal'].quantile(0.33)
q_kat_66 = master['rasio_KAT_terhadap_normal'].quantile(0.66)

def score_kat(x):
    if pd.isna(x):
        return np.nan
    elif x >= q_kat_66:
        return 0
    elif x >= q_kat_33:
        return 1
    else:
        return 2

master['KAT_score'] = master['rasio_KAT_terhadap_normal'].apply(score_kat)

print(q_kat_33, q_kat_66)

0.9475075591370681 1.0448487646227826


In [ ]:
#climate score

master['climate_score'] = (
    master['hujan_score'] +
    master['HTH_score'] +
    master['KAT_score']
)

In [ ]:
#historical score 2018 - 2024

hist = master[
    (master['tahun'] >= 2018) &
    (master['tahun'] <= 2024)
].copy()

annual = (
    hist.groupby(['wilayah', 'tahun'], as_index=False)
    .agg(
        produksi_padi_tahunan=('produksi_padi_bulanan', 'sum'),
        luas_panen_tahunan=('luas_panen_bulanan', 'sum')
    )
    .sort_values(['wilayah', 'tahun'])
)

annual['pct_change_produksi'] = annual.groupby('wilayah')['produksi_padi_tahunan'].pct_change()
annual['pct_change_luas_panen'] = annual.groupby('wilayah')['luas_panen_tahunan'].pct_change()

annual['turun_produksi'] = annual['pct_change_produksi'] <= -0.05
annual['turun_luas_panen'] = annual['pct_change_luas_panen'] <= -0.05
annual['indikasi_penurunan'] = annual['turun_produksi'] | annual['turun_luas_panen']

hist_profile = (
    annual.groupby('wilayah', as_index=False)
    .agg(
        freq_penurunan=('indikasi_penurunan', 'mean'),
        volatilitas_produksi=('pct_change_produksi', 'std'),
        volatilitas_luas_panen=('pct_change_luas_panen', 'std')
    )
)

hist_profile['volatilitas'] = hist_profile[
    ['volatilitas_produksi', 'volatilitas_luas_panen']
].mean(axis=1)

hist_profile['volatilitas'] = hist_profile['volatilitas'].fillna(0)

In [ ]:
q_freq_33 = hist_profile['freq_penurunan'].quantile(0.33)
q_freq_66 = hist_profile['freq_penurunan'].quantile(0.66)

q_vol_33 = hist_profile['volatilitas'].quantile(0.33)
q_vol_66 = hist_profile['volatilitas'].quantile(0.66)

def score_historical(row):
    high = (
        row['freq_penurunan'] >= q_freq_66 or
        row['volatilitas'] >= q_vol_66
    )

    medium = (
        row['freq_penurunan'] >= q_freq_33 or
        row['volatilitas'] >= q_vol_33
    )

    if high:
        return 2
    elif medium:
        return 1
    else:
        return 0

hist_profile['historical_score'] = hist_profile.apply(score_historical, axis=1)

In [ ]:
feature_table = master.merge(
    hist_profile[['wilayah', 'freq_penurunan', 'volatilitas', 'historical_score']],
    on='wilayah',
    how='left'
)

In [ ]:
#risk label

feature_table['final_score'] = (
    feature_table['climate_score'] +
    feature_table['historical_score']
)

def risk_label(score):
    if pd.isna(score):
        return np.nan
    elif score <= 2:
        return 'low'
    elif score <= 5:
        return 'medium'
    else:
        return 'high'

feature_table['risk_label'] = feature_table['final_score'].apply(risk_label)

In [ ]:
#target bulan depan

feature_table = feature_table.sort_values(['wilayah', 'tanggal'])

feature_table['risk_label_bulan_depan'] = (
    feature_table.groupby('wilayah')['risk_label'].shift(-1)
)

feature_table['final_score_bulan_depan'] = (
    feature_table.groupby('wilayah')['final_score'].shift(-1)
)

In [ ]:
#fitur lag

lag_cols = [
    'hujan_mm',
    'rasio_hujan_terhadap_normal',
    'HTH_hari',
    'rasio_KAT_terhadap_normal',
    'climate_score'
]

for col in lag_cols:
    feature_table[f'{col}_lag1'] = feature_table.groupby('wilayah')[col].shift(1)
    feature_table[f'{col}_roll3'] = (
        feature_table.groupby('wilayah')[col]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

In [ ]:
#faktor dominan

def dominant_factors(row):
    factors = {}

    if row['hujan_score'] > 0:
        factors['Curah hujan bawah normal'] = row['hujan_score']
    if row['HTH_score'] > 0:
        factors['HTH panjang'] = row['HTH_score']
    if row['KAT_score'] > 0:
        factors['Ketersediaan air rendah'] = row['KAT_score']
    if row['historical_score'] > 0:
        factors['Sensitivitas historis wilayah'] = row['historical_score']

    if len(factors) == 0:
        return 'Tidak ada faktor dominan ekstrem'

    max_score = max(factors.values())
    return ', '.join([k for k, v in factors.items() if v == max_score])

feature_table['faktor_dominan'] = feature_table.apply(dominant_factors, axis=1)

In [ ]:
def recommendation(row):
    if row['risk_label'] == 'high':
        return 'Prioritaskan pemantauan lapangan, cek irigasi/ketersediaan air, dan siapkan intervensi cepat.'
    elif row['risk_label'] == 'medium':
        return 'Lakukan monitoring berkala dan validasi kondisi lapangan pada wilayah rentan.'
    else:
        return 'Pantau rutin; belum membutuhkan intervensi prioritas.'

feature_table['rekomendasi_aksi'] = feature_table.apply(recommendation, axis=1)

In [ ]:
feature_table.to_csv('pantaupadi_feature_table_final.csv', index=False)

modeling_table = feature_table.dropna(subset=['risk_label_bulan_depan']).copy()
modeling_table.to_csv('pantaupadi_modeling_table_final.csv', index=False)

latest_2026 = feature_table[feature_table['tahun'] == 2026].copy()
latest_2026.to_csv('pantaupadi_latest_risk_2026_final.csv', index=False)

In [ ]:
#agar data yang tidak tersedia ga dipake

import numpy as np
import pandas as pd

feature_table = pd.read_csv('pantaupadi_feature_table_final.csv')

feature_table.loc[feature_table['tahun'].isin([2025, 2026]), 'produksi_padi_bulanan'] = np.nan

feature_table.loc[feature_table['tahun'] == 2026, 'luas_panen_bulanan'] = np.nan

feature_table.to_csv('pantaupadi_feature_table_final_clean.csv', index=False)

modeling_table = feature_table.dropna(subset=['risk_label_bulan_depan']).copy()
modeling_table.to_csv('pantaupadi_modeling_table_final_clean.csv', index=False)

latest_2026 = feature_table[feature_table['tahun'] == 2026].copy()
latest_2026.to_csv('pantaupadi_latest_risk_2026_final_clean.csv', index=False)